# 05 - Matrix Factorization

In [2]:
import sys
import os
os.chdir('/Users/mariaparis/Downloads/recommender_assignment_placeholders')
sys.path.insert(0, '/Users/mariaparis/Downloads/recommender_assignment_placeholders')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_ratings, load_items, train_test_split_ratings
from src.matrix_factorization import MatrixFactorizationRecommender

In [4]:
ratings = load_ratings()
items   = load_items()
train, test = train_test_split_ratings(ratings, test_size=0.2)
title_map = items.set_index(config.ITEM_COL)[config.TITLE_COL].to_dict()

## 1. How SVD works

We use the SVD algorithm from scikit-surprise. 
The model is trained by minimising regularised squared error using stochastic gradient descent.

**Design choices:**
- `n_factors=50`: 50 latent dimensions. Higher captures more nuance but risks overfitting.
- `n_epochs=20`: number of SGD passes over the training data.

## 2. Train the model

In [5]:
mf = MatrixFactorizationRecommender(n_factors=50, n_epochs=20, random_state=42)
mf.fit(train)
print("Model trained.")

Model trained.


## 3. Predict individual ratings

In [6]:
# Predict ratings for User 1 on a few specific movies
test_items = [
    (1,   "Toy Story (1995)"),
    (318, "Shawshank Redemption, The (1994)"),
    (296, "Pulp Fiction (1994)"),
    (356, "Forrest Gump (1994)"),
]

print("Predicted ratings for User 1:")
for movie_id, title in test_items:
    score = mf.predict_score(user_id=1, item_id=movie_id)
    print(f"  {title:<45} predicted: {score:.2f}")

Predicted ratings for User 1:
  Toy Story (1995)                              predicted: 3.50
  Shawshank Redemption, The (1994)              predicted: 3.50
  Pulp Fiction (1994)                           predicted: 3.50
  Forrest Gump (1994)                           predicted: 3.50


## 4. Recommendations for three users

In [7]:
for uid in [1, 50, 100]:
    recs = mf.recommend(uid, train, n=5)
    print(f"\nUser {uid}:")
    for rank, iid in enumerate(recs, 1):
        score = mf.predict_score(uid, iid)
        print(f"  {rank}. {title_map.get(iid, iid):<45} (predicted: {score:.2f})")


User 1:
  1. Secret Window (2004)                          (predicted: 3.50)
  2. Cove, The (2009)                              (predicted: 3.50)
  3. Indiana Jones and the Temple of Doom (1984)   (predicted: 3.50)
  4. Rocky II (1979)                               (predicted: 3.50)
  5. Newsies (1992)                                (predicted: 3.50)

User 50:
  1. Secret Window (2004)                          (predicted: 3.50)
  2. Cove, The (2009)                              (predicted: 3.50)
  3. Indiana Jones and the Temple of Doom (1984)   (predicted: 3.50)
  4. Abyss, The (1989)                             (predicted: 3.50)
  5. Rocky II (1979)                               (predicted: 3.50)

User 100:
  1. Secret Window (2004)                          (predicted: 3.50)
  2. Cove, The (2009)                              (predicted: 3.50)
  3. Indiana Jones and the Temple of Doom (1984)   (predicted: 3.50)
  4. Abyss, The (1989)                             (predicted: 3.50)
  5.

## 5. Effect of number of latent factors

More latent factors means the model can capture more complex patterns, but also increases the risk of overfitting. We train two versions to compare.

In [8]:
from src.evaluation import precision_at_k, recall_at_k, ndcg_at_k

eval_users = test[config.USER_COL].unique()[:50]
results = []

for n_factors in [10, 50, 100]:
    model = MatrixFactorizationRecommender(n_factors=n_factors, n_epochs=20, random_state=42)
    model.fit(train)
    
    precisions = []
    for uid in eval_users:
        user_test = test[test[config.USER_COL] == uid]
        relevant = set(user_test[user_test[config.RATING_COL] >= 3.5][config.ITEM_COL].values)
        if not relevant:
            continue
        recs = model.recommend(uid, train, n=10)
        precisions.append(precision_at_k(recs, relevant, k=10))
    
    results.append({'n_factors': n_factors, 'P@10': round(float(np.mean(precisions)), 4)})

pd.DataFrame(results)

,n_factors,P@10
0,10,0.032
1,50,0.032
2,100,0.032


## 6. Strengths and limitations

**Strengths:**
- Generalises across the entire user-item space, not just direct co-ratings
- Handles sparsity better than neighbourhood methods
- Bias terms explicitly model the fact that some users rate higher and some items are rated lower

**Limitations:**
- Still suffers from cold-start: new users or items with no ratings cannot be represented
- The latent factors are not interpretable: we cannot easily explain why an item was recommended
- Requires tuning of hyperparameters (n_factors, regularisation, learning rate)